# Reentrenar el extractor de comprobantes con un export más grande de Document AI

Flujo completo en Colab:
1. descarga del export de Document AI desde el bucket de Google Cloud Storage;
2. conversión al formato de docTR **conservando el split anterior** (los 589/81/168 documentos de `doctr.zip` se quedan en train/val/test; solo los nuevos se reparten);
3. evaluación del modelo actual (`lw_detr_s_fields_r3_photo`) sobre el split nuevo;
4. entrenamiento: se continúa desde `r3_photo` con la receta de su última etapa, ahora con más datos;
5. comparación en val, test una sola vez (total y solo los documentos nuevos) y prueba con una imagen.

**Por qué conservar el split.** `r3_photo` se entrenó con el train anterior. Si se rehace el split, algunos de esos comprobantes caerían en el test nuevo: el modelo actual parecería mejor de lo que es y, al seguir entrenando desde él, también el nuevo. Con el split conservado, el test nunca contiene una página que alguno de los dos modelos haya visto.

**Antes de empezar**
- `Entorno de ejecución > Cambiar tipo de entorno > GPU` (L4 o A100).
- En Drive, `MyDrive/doctr_data/`: `doctr.zip` (el dataset anterior) y `checkpoints/lw_detr_s_fields_r3_photo.pt` + `.json`.
- El export del dataset de Document AI en un bucket (Document AI > procesador > Dataset > Exportar). Cada JSON trae la imagen embebida y las entidades.

Todo lo que se produce (dataset, checkpoints, reportes) queda en Drive.

In [ ]:
# --- Parámetros: edita esta celda ---
GCP_PROJECT = ""  # id del proyecto de Google Cloud que tiene el bucket
GCS_EXPORT = "gs://<bucket>/<carpeta-del-export>/"  # carpeta del export de Document AI
DRIVE = "/content/drive/MyDrive/doctr_data"
PREVIOUS_ZIP = f"{DRIVE}/doctr.zip"  # dataset anterior: su split se conserva
DATASET = "doctr_v2"  # dataset nuevo: se guarda como {DRIVE}/{DATASET}.zip
TEST_RATIO, VAL_RATIO, SEED = 0.20, 0.12, 42  # para los documentos nuevos (las mismas proporciones de antes)
BASELINE = "lw_detr_s_fields_r3_photo"  # modelo actual en {DRIVE}/checkpoints
RUN = "lw_detr_s_fields_r4"  # modelo nuevo (A)
RUN_B = f"{RUN}_ema_map"  # variante opcional (B)
BRANCH = "finetune-invoice"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/fercho1999leon/doctr.git"
!test -d /content/doctr || git clone -q --branch $BRANCH $REPO /content/doctr
%cd /content/doctr
!git pull -q origin $BRANCH && git log --oneline -1
!pip install -q -e . tqdm matplotlib rapidfuzz
import torch

print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Descargar el export desde el bucket

Pide acceso con tu cuenta de Google (la que puede leer el bucket). `rsync` solo baja lo que falta: si se corta, vuelve a ejecutar la celda.

In [ ]:
import os

from google.colab import auth

auth.authenticate_user()
!gcloud config set project {GCP_PROJECT}
EXPORT_DIR = "/content/export"
os.makedirs(EXPORT_DIR, exist_ok=True)
!gcloud storage rsync -r "{GCS_EXPORT.rstrip('/')}" {EXPORT_DIR}
!echo "JSON descargados: $(find {EXPORT_DIR} -name '*.json' | wc -l)" && du -sh {EXPORT_DIR}

## 2. Convertir al formato de docTR

- `--keep-split`: los documentos del dataset anterior se quedan en el split que tenían (se reconocen por la imagen y por el id de Document AI). Los nuevos se reparten con `TEST_RATIO` / `VAL_RATIO`.
- `--group-by-key`: entre los nuevos, la misma transferencia subida dos veces (mismo número de comprobante) nunca queda repartida entre train y test.
- Si en Document AI alguna clase tiene otro nombre, añade `--rename viejo=nuevo` (por ejemplo `--rename facha=fecha`); para descartar una, `--drop clase`.

Si el zip del dataset nuevo ya está en Drive, se reutiliza (bórralo de Drive para regenerarlo).

In [ ]:
DATA = f"/content/datasets/{DATASET}"
ZIP = f"{DRIVE}/{DATASET}.zip"
PREVIOUS = "/content/datasets/previous/doctr"
LOG = f"{DATA}_convert.log"
sh = get_ipython().system
os.makedirs("/content/datasets", exist_ok=True)
sh(f'unzip -q -o "{PREVIOUS_ZIP}" -d /content/datasets/previous && ls {PREVIOUS}/*/manifest.json')
if os.path.exists(ZIP):
    print(f"Reutilizando {ZIP}")
    sh(f'unzip -q -o "{ZIP}" -d /content/datasets')
else:
    sh(
        f"python references/detection/convert_documentai.py --input {EXPORT_DIR} --output {DATA} "
        f"--keep-split {PREVIOUS} --test-ratio {TEST_RATIO} --val-ratio {VAL_RATIO} --seed {SEED} --group-by-key "
        f"> {LOG} 2>&1; tail -30 {LOG}"
    )
    # Documentos omitidos (sin imagen embebida, sin páginas): si son muchos, el export no es el correcto
    sh(f"echo omitidos: $(grep -c '^.skip.' {LOG}); grep '^.skip.' {LOG} | head -5")
    sh(f"grep -E '^Kept from' {LOG}; cp {LOG} '{DRIVE}/'")
    sh(f'cd /content/datasets && zip -qr "{ZIP}" {DATASET} && echo "Guardado en {ZIP}"')

In [ ]:
# Revisión del dataset antes de entrenar
import json

EXPECTED = {"cuenta_destino", "fecha", "nombre_cuenta_origen", "numero_comprobante", "numero_control", "valor_transferido"}
with open(f"{DATA}/audit.json", encoding="utf-8") as f:
    audit = json.load(f)
classes = set(audit["class_names"])
print("clases:", sorted(classes))
print("faltan:", sorted(EXPECTED - classes) or "-", "| sobran:", sorted(classes - EXPECTED) or "-")
for split, s in audit["splits"].items():
    print(f"\n{split}: {s['n_docs']} documentos")
    for c in sorted(classes):
        print(f"  {c:24s} {s['docs_with_class'][c]:5d} docs  ({s['presence_ratio'][c]:.0%})")
print("\nidentificadores compartidos entre splits:", len(audit["identifiers_shared_across_splits"]))


def split_ids(root, split):
    with open(f"{root}/{split}/manifest.json", encoding="utf-8") as f:
        return {d["id"] for d in json.load(f)["documents"]}


NEW_TEST = split_ids(DATA, "test") - split_ids(PREVIOUS, "test")
print(f"test: {len(split_ids(DATA, 'test'))} documentos, {len(NEW_TEST)} nuevos")

Qué revisar arriba:
- **Kept from ...**: train/val/test conservados deben ser 589/81/168 (o cerca) y *previous documents not found* cerca de 0. Si salen muchos no encontrados, el export nuevo no trae los documentos anteriores con la misma imagen ni el mismo id: avísame antes de entrenar.
- **faltan / sobran**: `-` en los dos. Si sobra una clase con nombre parecido (mal escrita en Document AI), reconvierte con `--rename` (borra antes el zip de Drive).
- **identificadores compartidos entre splits**: idealmente 0. Si aparecen, son la misma transferencia en train y test.

## 3. Modelo actual sobre el split nuevo

Es la referencia. Todas las evaluaciones usan el mismo postproceso (`POST`).

In [ ]:
CKPT = f"{DRIVE}/checkpoints"
EVAL = f"{DRIVE}/eval/{DATASET}"
os.makedirs(EVAL, exist_ok=True)
POST = "--top-k-per-class 1 --merge-fragments --fallback --straighten"


def evaluate(name, split="val"):
    get_ipython().system(
        f"python references/detection/evaluate_fields.py --checkpoint {CKPT}/{name}.pt --data {DATA}/{split} "
        f"{POST} --output {EVAL}/{name}_{split} | tail -25"
    )


def report(name, split="val"):
    with open(f"{EVAL}/{name}_{split}.json", encoding="utf-8") as f:
        return json.load(f)


evaluate(BASELINE, "val")

## 4. Entrenar

**A (recomendado): continuar desde `r3_photo`** con la receta de su última etapa (`--resume`, lr 2e-4, `--photo-aug --perspective 0.2`, batch 8, 768 px) sobre el train ampliado. La celda siguiente comprueba que `BASE` coincide con los argumentos guardados en `{BASELINE}.json` y avisa de cualquier diferencia.

Duración: la última vez cada época con 589 imágenes tardó ~70 s en train + 4 s en val; crece en proporción al train nuevo. Con early stopping (paciencia 12) suele parar antes de 40 épocas. `train()` no reentrena si el checkpoint ya existe en Drive: tras una desconexión, borra `{RUN}.pt` para empezar de cero, o continúa desde la última época guardada con `--resume {CKPT}/{RUN}_epochN.pt`.

In [ ]:
import importlib.util
import shlex
import sys

# Receta de la última etapa de r3_photo (sin --resume: lo añade train())
BASE = (
    "--labels-name labels_layout.json --device 0 --amp --amp-dtype bfloat16 -b 8 --input_size 768 "
    "--epochs 40 --lr 2e-4 --backbone-lr 2e-4 --no-hflip --perspective 0.2 --photo-aug "
    "--early-stop --early-stop-epochs 12 --early-stop-delta 0.001 --save-interval-epoch -j 4"
)


def parsed_options(extra=""):
    """The options train.py would run with, parsed by train.py itself."""
    spec = importlib.util.spec_from_file_location("layout_train", "references/layout/train.py")
    module = importlib.util.module_from_spec(spec)
    sys.path.insert(0, "references/layout")
    try:
        spec.loader.exec_module(module)
        argv = ["train.py", "lw_detr_s", *shlex.split(f"{BASE} {extra}"), "--train_path", "-", "--val_path", "-"]
        saved, sys.argv = sys.argv, argv
        try:
            return vars(module.parse_args())
        finally:
            sys.argv = saved
    finally:
        sys.path.remove("references/layout")


IGNORED = {"train_path", "val_path", "output_dir", "name", "device", "workers", "resume", "wb", "clearml"}
with open(f"{CKPT}/{BASELINE}.json", encoding="utf-8") as f:
    base_args = json.load(f)["args"]
ours = parsed_options()
diff = {k: (base_args[k], v) for k, v in ours.items() if k in base_args and k not in IGNORED and base_args[k] != v}
print("BASE coincide con la receta de r3_photo" if not diff else "Distinto de r3_photo (antes -> ahora):")
for k, (old, new) in diff.items():
    print(f"  {k}: {old!r} -> {new!r}")

In [ ]:
def train(name, extra=""):
    if os.path.exists(f"{CKPT}/{name}.pt"):
        print(f"{name}.pt ya existe: se reutiliza (bórralo para reentrenar)")
        return
    get_ipython().system(
        f"python references/layout/train.py lw_detr_s {BASE} {extra} --train_path {DATA}/train "
        f"--val_path {DATA}/val --output_dir {CKPT} --name {name}"
    )


# A: misma receta, más datos, desde r3_photo
train(RUN, f"--resume {CKPT}/{BASELINE}.pt")
evaluate(RUN, "val")

### Opcional: B, igual que A con EMA y selección por mAP

`--ema` promedia los pesos durante el entrenamiento y `--select-by map` guarda la época con mejor mAP en vez de la de menor pérdida. No están medidos en estos datos: B se queda solo si mejora a A en `val`. Cuesta otro entrenamiento completo.

In [ ]:
train(RUN_B, f"--resume {CKPT}/{BASELINE}.pt --ema --select-by map")
evaluate(RUN_B, "val")

## 5. Comparar en val

Acierto de un valor por campo (lo que recibe el ERP) y documentos con todos los campos requeridos correctos. `vs base` cuenta los documentos que solo uno de los dos acierta, con la p de McNemar: con p > 0.05 la diferencia puede ser ruido.

In [ ]:
import pandas as pd
from scipy.stats import binomtest


def compare(names, split="val", base=BASELINE, ids=None):
    """Accuracy of the one value per field, over every document of the split or only `ids`."""

    def docs(name):
        return {d["id"]: d for d in report(name, split)["per_document"] if ids is None or d["id"] in ids}

    ref = docs(base)
    rows = []
    for name in [base, *names]:
        if not os.path.exists(f"{EVAL}/{name}_{split}.json"):
            print(f"{name}: sin reporte todavía")
            continue
        run = docs(name)
        row = {"modelo": name, "todos los requeridos": f"{sum(d['all_required_value_ok'] for d in run.values())}/{len(run)}"}
        for c in next(iter(run.values()))["fields"]:
            row[c] = f"{sum(d['fields'][c]['value_ok'] for d in run.values()) / len(run):.1%}"
        if name != base:
            only_base = sum(ref[k]["all_required_value_ok"] and not run[k]["all_required_value_ok"] for k in ref)
            only_run = sum(run[k]["all_required_value_ok"] and not ref[k]["all_required_value_ok"] for k in ref)
            n = only_base + only_run
            p = binomtest(only_run, n, 0.5).pvalue if n else 1.0
            row["vs base"] = f"+{only_run} / -{only_base} (p={p:.2f})"
        rows.append(row)
    return pd.DataFrame(rows).set_index("modelo")


compare([RUN, RUN_B])

## 6. Test, una sola vez, con el modelo elegido

Elige en `val` (A o B) y escribe su nombre en `FINAL`. El test se mira una vez: si después de verlo cambias algo y vuelves a mirar, deja de ser una medida limpia. La segunda tabla mide solo los documentos de test nuevos (los que no estaban en el test anterior): es lo más parecido a lo que llega a producción ahora.

In [ ]:
FINAL = RUN  # o RUN_B si ganó en val
evaluate(BASELINE, "test")
evaluate(FINAL, "test")
display(compare([FINAL], split="test"))
display(compare([FINAL], split="test", ids=NEW_TEST))

## 7. Probar con una imagen y entregar

`--format receipt` devuelve el contrato tipado (fecha ISO, monto decimal, cuenta enmascarada) de `references/detection/receipt_schema.py`.

In [ ]:
img = sorted(os.listdir(f"{DATA}/test/images"))[0]
!python references/detection/kie_inference.py --checkpoint {CKPT}/{FINAL}.pt {POST} --format receipt --json /content/prueba.json "{DATA}/test/images/{img}" --quiet
!head -c 1500 /content/prueba.json

Para desplegar el modelo nuevo en el servicio OCR del ERP (proveedor `doctr_fields`), copia **los dos archivos** de `MyDrive/doctr_data/checkpoints/`: `<FINAL>.pt` (pesos) y `<FINAL>.json` (arquitectura y nombres de clase). Sin el `.json` el modelo no carga.

Al terminar: `Entorno de ejecución > Desconectar y borrar el entorno de ejecución` para no seguir gastando unidades.